# Human approval with interrupts

Read the [guide](README.md). Run cells in order with the notebooks and apps groups. Offline replay uses real SDK and local protocol paths; it does not measure live model quality.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_hook_interrupt

Predict the outcome, then run the [script](examples/01_hook_interrupt.py).


In [ ]:
"""Hook interrupt: Nimbus Tier 3 lesson.

Run: uv run python modules/12_interrupts/examples/01_hook_interrupt.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from pathlib import Path
from tempfile import TemporaryDirectory

from common.approvals import ApprovalLedger, approval_agent, request_prompt, responses_for
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--decision", choices=["approve", "reject", "expire", "ask"], default="reject"
    )
    parser.add_argument("--action", choices=["refund", "change"], default="refund")
    args, cfg = lesson_settings(parser)
    with TemporaryDirectory() as directory:
        ledger = ApprovalLedger(Path(directory) / "approval.db")
        key = ledger.propose(
            "CUS-001", "DEMO-001", args.action, 60000 if args.action == "refund" else 0
        )
        agent = approval_agent(cfg, ledger, key)
        result = agent(request_prompt(ledger, key), limits={"turns": 3})
        if result.stop_reason == "interrupt":
            print("Review exact terms:", result.interrupts[0].reason)
            decision = (
                input("approve/reject: ").strip() if args.decision == "ask" else args.decision
            )
            expired = ledger.get(key, "CUS-001")["expires"] if decision == "expire" else None
            ledger.decide(
                key,
                "CUS-001",
                "approve" if decision in {"approve", "expire"} else "reject",
                supervisor="SUPERVISOR-DEMO",
                now=expired,
            )
            result = agent(responses_for(result), limits={"turns": 3})
        print("Authoritative status:", ledger.get(key, "CUS-001")["status"])
        complete(result, cfg)


run_lesson(main, "modules/12_interrupts/examples/01_hook_interrupt.py")

## 02_tool_interrupt

Predict the outcome, then run the [script](examples/02_tool_interrupt.py).


In [ ]:
"""Tool interrupt: Nimbus Tier 3 lesson.

Run: uv run python modules/12_interrupts/examples/02_tool_interrupt.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

from strands import Agent, tool
from strands.types.tools import ToolContext

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


def main():
    _, cfg = lesson_settings()

    @tool(context=True)
    def choose_contact(tool_context: ToolContext) -> str:
        """Ask a human for a preferred contact channel; does not authorize money movement."""
        answer = tool_context.interrupt("contact-choice", reason="Choose email or phone")
        return answer if answer in {"email", "phone"} else "No valid choice"

    agent = Agent(
        model=get_model(cfg, responses=[tool_message("choose_contact", {}), "Choice recorded."]),
        tools=[choose_contact],
        callback_handler=None,
        retry_strategy=None,
    )
    result = agent("Ask for a contact preference", limits={"turns": 3})
    if result.stop_reason == "interrupt":
        result = agent(
            [{"interruptResponse": {"interruptId": result.interrupts[0].id, "response": "email"}}]
        )
    # Simpler console handoff is supported, but it is not a durable approval system.
    from unittest.mock import patch

    from strands_tools import handoff_to_user

    with patch("strands_tools.handoff_to_user.get_user_input", return_value="email"):
        simple = Agent(
            model=get_model(cfg, responses=[]), tools=[handoff_to_user], callback_handler=None
        )
        print(
            simple.tool.handoff_to_user(message="Choose a contact channel", breakout_of_loop=False)
        )
    complete(result, cfg)


run_lesson(main, "modules/12_interrupts/examples/02_tool_interrupt.py")

## 03_approval_cli

Predict the outcome, then run the [script](examples/03_approval_cli.py).


In [ ]:
"""Approval cli: Nimbus Tier 3 lesson.

Run: uv run python modules/12_interrupts/examples/03_approval_cli.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from pathlib import Path
from tempfile import TemporaryDirectory

from common.approvals import ApprovalLedger, approval_agent, request_prompt, responses_for
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--decision", choices=["approve", "reject", "expire", "ask"], default="reject"
    )
    parser.add_argument("--action", choices=["refund", "change"], default="refund")
    args, cfg = lesson_settings(parser)
    with TemporaryDirectory() as directory:
        ledger = ApprovalLedger(Path(directory) / "approval.db")
        key = ledger.propose(
            "CUS-001", "DEMO-001", args.action, 60000 if args.action == "refund" else 0
        )
        agent = approval_agent(cfg, ledger, key)
        result = agent(request_prompt(ledger, key), limits={"turns": 3})
        if result.stop_reason == "interrupt":
            print("Review exact terms:", result.interrupts[0].reason)
            decision = (
                input("approve/reject: ").strip() if args.decision == "ask" else args.decision
            )
            expired = ledger.get(key, "CUS-001")["expires"] if decision == "expire" else None
            ledger.decide(
                key,
                "CUS-001",
                "approve" if decision in {"approve", "expire"} else "reject",
                supervisor="SUPERVISOR-DEMO",
                now=expired,
            )
            result = agent(responses_for(result), limits={"turns": 3})
        print("Authoritative status:", ledger.get(key, "CUS-001")["status"])
        complete(result, cfg)


run_lesson(main, "modules/12_interrupts/examples/03_approval_cli.py")

## 04_approval_web

Predict the outcome, then run the [script](examples/04_approval_web.py).


In [ ]:
"""Approval web: Nimbus Tier 3 lesson.

Run: uv run --group apps python modules/12_interrupts/examples/04_approval_web.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from pathlib import Path
from tempfile import TemporaryDirectory

from common.approvals import ApprovalLedger
from common.config import ROOT
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.web_apps import demonstrate, serve


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--serve", action="store_true")
    args, cfg = lesson_settings(parser)
    if args.serve:
        serve(cfg, ledger=ApprovalLedger(ROOT / ".nimbus/approval_web.db"))
    else:
        with TemporaryDirectory() as directory:
            demonstrate(cfg, ledger=ApprovalLedger(Path(directory) / "approval.db"))
        complete()


run_lesson(main, "modules/12_interrupts/examples/04_approval_web.py")

## 05_batch_interrupt

Predict the outcome, then run the [script](examples/05_batch_interrupt.py).


In [ ]:
"""Pause a whole tool batch before any tool runs.

Run: uv run python modules/12_interrupts/examples/05_batch_interrupt.py --mock
Expected: zero tools before review, two after approval, Lesson complete.
Cost: offline $0; live inference uses configured rates. AWS resources: none.
"""

from strands import Agent, tool
from strands.hooks import BeforeToolsEvent, HookProvider

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


class ReviewBatch(HookProvider):
    def register_hooks(self, registry, **kwargs):
        registry.add_callback(BeforeToolsEvent, self.review)

    def review(self, event):
        names = [b["toolUse"]["name"] for b in event.message["content"] if "toolUse" in b]
        response = event.interrupt("review-batch", reason={"tools": names})
        if response != "approve":
            event.cancel = "Batch rejected"


def main():
    _, cfg = lesson_settings()
    calls = []

    @tool
    def inspect_fare(code: str) -> str:
        """Inspect one synthetic fare without changing a booking."""
        calls.append(code)
        return f"{code}: non-refundable"

    batch = tool_message("inspect_fare", {"code": "A"}, "fare-a")
    batch["content"] += tool_message("inspect_fare", {"code": "B"}, "fare-b")["content"]
    agent = Agent(
        model=get_model(cfg, responses=[batch, "Both fares inspected."]),
        tools=[inspect_fare],
        hooks=[ReviewBatch()],
        callback_handler=None,
    )
    result = agent("Inspect fares A and B together.", limits={"turns": 3})
    print("Tools before review:", len(calls))
    if result.stop_reason == "interrupt":
        # This host-generated approval is only for harmless synthetic reads.
        # Financial actions use the durable ledger and authenticated supervisor in example 01.
        result = agent(
            [
                {"interruptResponse": {"interruptId": item.id, "response": "approve"}}
                for item in result.interrupts
            ],
            limits={"turns": 3},
        )
    print("Tools after review:", len(calls))
    complete(result, cfg)


run_lesson(main, "modules/12_interrupts/examples/05_batch_interrupt.py")

## refund_approval

Predict the outcome, then run the [script](use_case/refund_approval.py).


In [ ]:
"""Und approval: Nimbus Tier 3 lesson.

Run: uv run python modules/12_interrupts/use_case/refund_approval.py --mock
Expected: Lesson complete. Offline fixtures cost $0.
Live inference uses configured rates. AWS resources created by this script: none.
"""

import argparse
from pathlib import Path
from tempfile import TemporaryDirectory

from common.approvals import ApprovalLedger, approval_agent, request_prompt, responses_for
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--decision", choices=["approve", "reject", "expire", "ask"], default="reject"
    )
    parser.add_argument("--action", choices=["refund", "change"], default="refund")
    args, cfg = lesson_settings(parser)
    with TemporaryDirectory() as directory:
        ledger = ApprovalLedger(Path(directory) / "approval.db")
        key = ledger.propose(
            "CUS-001", "DEMO-001", args.action, 60000 if args.action == "refund" else 0
        )
        agent = approval_agent(cfg, ledger, key)
        result = agent(request_prompt(ledger, key), limits={"turns": 3})
        if result.stop_reason == "interrupt":
            print("Review exact terms:", result.interrupts[0].reason)
            decision = (
                input("approve/reject: ").strip() if args.decision == "ask" else args.decision
            )
            expired = ledger.get(key, "CUS-001")["expires"] if decision == "expire" else None
            ledger.decide(
                key,
                "CUS-001",
                "approve" if decision in {"approve", "expire"} else "reject",
                supervisor="SUPERVISOR-DEMO",
                now=expired,
            )
            result = agent(responses_for(result), limits={"turns": 3})
        print("Authoritative status:", ledger.get(key, "CUS-001")["status"])
        complete(result, cfg)


run_lesson(main, "modules/12_interrupts/use_case/refund_approval.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
